In [ ]:
import subprocess, torch
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.run(['pip','install','-q','diffusers','transformers','accelerate'])
import urllib.request
from PIL import Image, ImageOps, ImageDraw, ImageFilter
import numpy as np

# בסיס החזה המאושר של סאן (802-1)
BASE = Image.open(urllib.request.urlopen('https://aidoor.shimi.land/gallery/katie-chest-17-10/inp_s802.png')).convert('RGB')
W, H = BASE.size
print('base', BASE.size, flush=True)

from diffusers import StableDiffusionXLInpaintPipeline, StableDiffusionXLImg2ImgPipeline
pipe = StableDiffusionXLInpaintPipeline.from_pretrained('SG161222/RealVisXL_V4.0', torch_dtype=torch.float16, variant='fp16').to('cuda')
pipe.enable_attention_slicing()
print('pipe ready', flush=True)

# בלוק זהות נעול — בסטייל הפוקר, מילה במילה בכל רנדר
LOCK = ("the exact same woman as the input, identical face, identical dark hair in a bun, "
"fitness trainer physique, solid athletic toned body, low body fat, toned flat stomach, "
"large full natural perky bust with prominent volume and upper fullness, deep cleavage, "
"rounded firm glutes, slim waist, ")
APPAREL = ("topless with bare breasts fully exposed, no clothing, no underwear, no bra, "
"wearing only a thin sheer floral lace thong, ")
STYLE = ("photorealistic photograph, realistic skin texture, sharp focus, studio lighting, film grain, ")
NEG = ("bra, bikini top, tank top, sports bra, crop top, clothing on chest, covered breasts, underwear top, "
"fat, chubby, overweight, belly, pot belly, thick waist, small breasts, flat chest, "
"different face, different woman, extra breasts, deformed nipples, distorted anatomy, extra limbs, "
"blurry, cartoon, anime, worst quality, low quality, watermark, text")

# שלב 1: החלפת לבוש לבגד טסט סינתטי (inpaint על אזור החזה-גו בלבד)
mask = Image.new('L', (W, H), 0)
d = ImageDraw.Draw(mask)
cx, cy = W//2, int(H*0.42)
rx, ry = int(W*0.24), int(H*0.13)
d.ellipse([cx-rx, cy-ry, cx+rx, cy+ry], fill=255)
mask = mask.filter(ImageFilter.GaussianBlur(12))
mask.save('/kaggle/working/mask.png')
P1 = LOCK + APPAREL + "same pose as input, " + STYLE
s1 = pipe(P1, negative_prompt=NEG, image=BASE, mask_image=mask,
          strength=0.80, guidance_scale=7.5, num_inference_steps=40, seed=8211).images[0]
s1 = pipe(P1, negative_prompt=NEG, image=s1, mask_image=mask,
          strength=0.35, guidance_scale=7.0, num_inference_steps=28, seed=8212).images[0]
s1.save('/kaggle/working/ktest_apparel_base.png')
print('apparel base done', flush=True)

del pipe
import gc; gc.collect(); torch.cuda.empty_cache()
pipe2 = StableDiffusionXLImg2ImgPipeline.from_pretrained('SG161222/RealVisXL_V4.0', torch_dtype=torch.float16, variant='fp16').to('cuda')
pipe2.enable_attention_slicing()

# שלב 2: גוף מלא ראש עד רגליים — קדימה ואחורה (פורמט מבחן), 2 זרעים כל אחד
FRONT = ("full body photograph, head to toe, standing pose facing the camera, arms relaxed at her sides, " + LOCK + APPAREL + STYLE)
BACK = ("full body photograph, head to toe, standing pose seen from behind, back to the camera, " + LOCK + APPAREL + STYLE)
for tag, pr in [('front', FRONT), ('back', BACK)]:
    for seed in [8301, 8302]:
        img = pipe2(pr, negative_prompt=NEG, image=s1, strength=0.55, guidance_scale=7.0,
                    num_inference_steps=36, seed=seed).images[0]
        img.save(f'/kaggle/working/ktest_full_{tag}_s{seed}.png')
        print('full', tag, seed, 'done', flush=True)

# שלב 3: מבחן פיזיקה — רכינה קדימה 45 מעלות, החזה נופל בכבידה
GRAV = ("leaning forward toward the camera about 45 degrees, hands on her thighs, "
        "her large heavy natural bare breasts hanging straight down under full gravity, "
        "deep natural sag and droop, elongated heavy teardrop shape pulled down by gravity, " + LOCK + APPAREL + STYLE)
GNEG = NEG + ", floating breasts, perky uplifted breasts"
for seed in [8401]:
    img = pipe2(GRAV, negative_prompt=GNEG, image=s1, strength=0.55, guidance_scale=7.0,
                num_inference_steps=36, seed=seed).images[0]
    img.save(f'/kaggle/working/ktest_grav_s{seed}.png')
    print('gravity done', flush=True)

# וריאציות שחור-לבן לכל הפלטים
import glob
for f in glob.glob('/kaggle/working/ktest_*.png'):
    im = Image.open(f)
    ImageOps.autocontrast(im.convert('L'), cutoff=1).convert('RGB').save(f.replace('.png','_bw.png'))
print('bw done', flush=True)

b = np.asarray(BASE, dtype=np.float32)
m = np.asarray(mask, dtype=np.float32)/255.0
a = np.asarray(Image.open('/kaggle/working/ktest_apparel_base.png').convert('RGB'), dtype=np.float32)
diff = np.abs(a-b).mean(axis=2)
print('VALIDATION apparel: in-mask', round(float((diff*m).sum()/m.sum()),2),
      'out-mask', round(float((diff*(1-m)).sum()/(1-m).sum()),2), flush=True)
print('ALL DONE', flush=True)